<a href="https://colab.research.google.com/github/rizzken/MedRAG-Eval/blob/main/01_rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG Pipeline: Spinal Health Knowledge Base
*For detailed evaluation results see 02_ragas_evaluation.ipynb and 03_framework_comparison.ipynb*

## Environment Setup
Installing required packages for RAG pipeline.
Note: `langchain-google-vertexai` is installed here as a dependency patch
required by RAGAS in subsequent notebooks — not used in this pipeline.

In [1]:
# Base packages for this section
!pip install -qU langchain-core langchain-text-splitters langchain-community langchain-chroma langchain-ollama beautifulsoup4 lxml

# If you hit: deepeval requires click<8.4.0 but click 8.4.1 is installed
# run the three commands below to fix and verify dependency conflicts.
!python -m pip install "click>=8.0.0,<8.4.0"
!python -m pip install --upgrade --force-reinstall "deepeval==4.0.4"
!python -m pip check

In [2]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
from langchain_ollama import ChatOllama

/tmp/ipykernel_565/2351560790.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


## Model Setup
Installing and starting Ollama server.

**Generator model:** `qwen2.5:7b` — chosen over 3B for medical domain accuracy.  
**Embedding model:** `nomic-embed-text` — lightweight, works well with Chroma locally.

*Both models run on-device within the Colab runtime — no external API calls or cloud LLM services used.*

In [3]:
!apt-get install -y zstd
# Install ollama
!curl -fsSL https://ollama.com/install.sh | sh

!pip install ollama

# Start ollama server in the background
!ollama serve &> ollama.log &

import time
time.sleep(10)

print("Ollama installed and server started")

!ollama pull qwen2.5:7b
!ollama pull nomic-embed-text:latest


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (4,692 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 118242 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current u

In [4]:
llm = ChatOllama(
    base_url="http://localhost:11434",
    model = "qwen2.5:7b",
    temperature=0,
    max_tokens = 200
)

## RAG Pipeline Construction

This cell builds the complete pipeline in four steps:

1. **Data ingestion** — loads 4 WebMD articles on spinal health topics
2. **Chunking** — splits text into 500-character chunks (100-char overlap to preserve context at boundaries)
3. **Vector store** — embeds chunks with `nomic-embed-text` and stores in Chroma
4. **Retrieval chain** — fetches top 10 relevant chunks per query, passes to `qwen2.5:7b` with a strict prompt

Note: `langchain-google-vertexai` is installed as a compatibility patch for RAGAS — not used in this pipeline.

In [5]:
!pip install langchain-google-vertexai -q

# Load data from Web
loader = WebBaseLoader(["https://www.webmd.com/cancer/what-is-chordoma",
                        "https://www.webmd.com/brain/spinal-muscular-atrophy",
                        "https://www.webmd.com/back-pain/spinal-stenosis",
                        "https://www.webmd.com/pain-management/pain-management-treatment-overview"])
data = loader.load()

# Split text into documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(data)

# Add text to vector db
embedding = OllamaEmbeddings(model="nomic-embed-text:latest")
vectordb = Chroma.from_documents(documents=splits, embedding=embedding)

# Create a retriever
retriever = vectordb.as_retriever(
    search_kwargs={"k": 10}
)

def format_docs(docs: List[Document]) -> str:
    return "\n\n".join([d.page_content for d in docs])


template = """Answer the question based only on the following context:

    {context}

    Be accurate and specific. If the answer is not in the context, say you don't know.

    Question: {question}
    """
prompt = ChatPromptTemplate.from_template(template)


def retrieve_and_format(question):
    docs = retriever.invoke(question)
    return format_docs(docs)

chain = {"context": retrieve_and_format, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.7/118.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.8/354.8 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.3/66.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 934.9/934.9 kB 13.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.40.0 requires rich<14,>

## Pipeline Verification
Quick sanity check — verifying the pipeline returns a coherent answer
before running full evaluation.

In [6]:
response = chain.invoke("What is SMA?")

print(response)

According to the provided context, Spinal Muscular Atrophy (SMA) is a condition characterized by:

- **Symptoms**: The document mentions "Spinal Muscular Atrophy Symptoms," indicating that symptoms are present but does not list them specifically.
- **Causes**: It states "Spinal Muscular Atrophy Causes," suggesting that the causes of SMA are discussed, though specific causes are not detailed in this context.
- **Diagnosis**: There is mention of "How Is Spinal Muscular Atrophy Diagnosed?" implying that diagnostic methods exist but they are not elaborated upon here.
- **Treatment**: The document includes "Spinal Muscular Atrophy Treatment," indicating that treatment options are available, though the specifics are not provided in this context.

For more detailed information on symptoms, causes, diagnosis, and treatment of SMA, you would need to refer to a comprehensive medical source or consult with a healthcare professional.


## Summary

Pipeline successfully built and verified.

**Key decisions:**
- `qwen2.5:7b` selected over smaller models for medical domain accuracy
- `k=10` retriever returns more chunks to reduce risk of missing relevant context
- Strict prompt instructs the model to acknowledge knowledge gaps rather than hallucinate

**Next steps:** see `02_ragas_evaluation.ipynb` for automated quality evaluation using RAGAS metrics.